# Pengenalan Vokal Bahasa Indonesia

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config.yaml').is_file():
    raise FileNotFoundError('Buka notebook ini dari folder root proyek.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from vowel_recognition.config import load_config
from vowel_recognition.data_loader import build_manifest, validate_dataset, dataset_summary
from vowel_recognition.preprocessing import preprocess_audio
from vowel_recognition.features import extract_features
from vowel_recognition.templates import build_templates, save_templates
from vowel_recognition.gaussian_model import build_models
from vowel_recognition.recognition import predict_batch
from vowel_recognition.evaluation import evaluate
from vowel_recognition.visualization import save_confusion, save_accuracy_comparison


## 1. Muat konfigurasi dan validasi dataset

In [ ]:
config = load_config(PROJECT_ROOT / 'config.yaml')
manifest, validation_errors = validate_dataset(config)
print(dataset_summary(manifest, validation_errors))
display(manifest)
if validation_errors:
    print('\nEksperimen belum dijalankan. Lengkapi rekaman berikut terlebih dahulu:')
    print(*validation_errors, sep='\n')
else:
    print('Dataset valid dan siap diproses.')

## 2. Pemuatan audio, prapemrosesan, dan MFCC


In [ ]:
if not validation_errors:
    example = manifest.iloc[0]
    audio, sample_rate = preprocess_audio(example.file_path, config)
    mfcc_features = extract_features(audio, sample_rate, config)
    print(f"File: {example.file_path}\nSample rate: {sample_rate} Hz\nMFCC: {mfcc_features.shape}")
    plt.figure(figsize=(12, 3))
    plt.plot(audio)
    plt.title('Gelombang audio setelah prapemrosesan')
    plt.xlabel('Sampel')
    plt.ylabel('Amplitudo')
    plt.show()
else:
    print('Lewati prapemrosesan contoh karena dataset belum lengkap.')

## 3. Jalankan eksperimen DTW dan Gaussian

In [ ]:
metrics = None
artifacts = None

if not validation_errors:
    np.random.seed(config.get('random_seed', 42))
    results_dir = Path(config['results']['dir'])
    tables_dir = results_dir / 'tables'
    figures_dir = results_dir / 'figures'
    cache_dir = results_dir / 'feature_cache'
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    def load_features(file_path: str) -> np.ndarray:
        relative_path = Path(file_path).relative_to(Path(config['data']['directory']))
        cache_path = (cache_dir / relative_path).with_suffix('.npy')
        if config.get('cache_features', True) and cache_path.is_file():
            return np.load(cache_path)
        audio, rate = preprocess_audio(file_path, config)
        features = extract_features(audio, rate, config)
        if config.get('cache_features', True):
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_path, features)
        return features

    train_manifest = manifest[manifest.split == 'train'].copy()
    training_features = {row.file_path: load_features(row.file_path) for row in train_manifest.itertuples()}
    features_by_vowel = {
        vowel: [training_features[row.file_path] for row in train_manifest[train_manifest.true_vowel == vowel].itertuples()]
        for vowel in config['data']['vowels']
    }
    artifacts = {
        'dtw': build_templates(features_by_vowel, config['dtw']['normalize']),
        'gaussian': build_models(features_by_vowel, config['gaussian']['regularization']),
    }
    save_templates(artifacts['dtw'], results_dir / 'artifacts')

    scenarios = {'closed_recording': train_manifest, 'closed_speakers': manifest[manifest.split == 'test']}
    open_speakers = config['data'].get('open_test_speakers', [])
    if open_speakers:
        scenarios['open_speakers'] = build_manifest(config, open_speakers, ('test',))

    metrics = {}
    for system in ('dtw', 'gaussian'):
        metrics[system] = {}
        for scenario, scenario_manifest in scenarios.items():
            predictions = predict_batch(scenario_manifest, system, artifacts[system], config)
            pd.DataFrame(predictions).to_csv(tables_dir / f'predictions_{system}_{scenario}.csv', index=False)
            metric = evaluate(predictions, config['data']['vowels'])
            metrics[system][scenario] = metric
            save_confusion(metric, f'{system} - {scenario}', figures_dir / f'confusion_{system}_{scenario}.png')
        if 'open_speakers' not in metrics[system]:
            metrics[system]['open_speakers'] = evaluate([], config['data']['vowels'])

    comparison_rows = []
    for system, scenario_metrics in metrics.items():
        available_scores = [metric['accuracy'] for metric in scenario_metrics.values() if metric['available']]
        comparison_rows.append({
            'System': system,
            'Closed Recording (%)': scenario_metrics['closed_recording']['accuracy'],
            'Closed Speakers (%)': scenario_metrics['closed_speakers']['accuracy'],
            'Open Speakers (%)': scenario_metrics['open_speakers']['accuracy'],
            'Average (%)': float(np.mean(available_scores)) if available_scores else None,
        })
    pd.DataFrame(comparison_rows).to_csv(tables_dir / 'comparison.csv', index=False)
    (results_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    (results_dir / 'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    if config.get('generate_plots', True):
        save_accuracy_comparison(metrics, figures_dir / 'accuracy_comparison.png')
    print('Selesai. Output disimpan di:', results_dir)

else:
    print('Eksperimen tidak dijalankan sampai semua rekaman valid.')

## 4. Tampilkan dan simpan hasil

In [ ]:
results_dir = Path(config['results']['dir'])
comparison_path = results_dir / 'tables' / 'comparison.csv'
metrics_path = results_dir / 'metrics.json'
if comparison_path.is_file():
    display(pd.read_csv(comparison_path))
if metrics_path.is_file():
    saved_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(json.dumps(saved_metrics, indent=2))
print('\nTabel tersimpan:')
for path in sorted((results_dir / 'tables').glob('*.csv')):
    print('-', path.relative_to(PROJECT_ROOT))
print('\nGambar tersimpan:')
for path in sorted((results_dir / 'figures').glob('*.png')):
    print('-', path.relative_to(PROJECT_ROOT))